# Step 03 — Viterbi Decoding
**NLP Course Project · Statistical POS Tagging using HMMs**  
23BCS052 · Hammad Malik

---

### What this notebook does
Takes the trained HMM from Step 02 and uses the **Viterbi algorithm** to tag brand new sentences.

> 🧠 **Mental model:** Think of Viterbi as filling a grid.  
> - Each **column** = one word in the sentence  
> - Each **row** = one possible POS tag  
> - Each **cell** = the best score for being at that tag at that word position  
>  
> We fill left → right, then backtrack right → left to read off the best tag path.

```
        "The"      "dog"      "runs"     "fast"
  NN  [ -12.3  ] [ -8.1  ] [ -9.4  ] [ -11.2 ]
  VB  [ -18.6  ] [ -15.3 ] [ -6.2  ] [ -9.8  ]   ← best path
  DT  [  -4.1  ] [ -14.7 ] [ -16.1 ] [ -14.3 ]
  JJ  [ -16.2  ] [ -11.5 ] [ -13.3 ] [ -7.1  ]
  ...
```

---
## Cell 1 — Imports & Load the HMM

In [ ]:
import pickle
import numpy as np

# Load the trained HMM from Step 02
with open('hmm_model.pkl', 'rb') as f:
    hmm = pickle.load(f)

log_A   = hmm['log_A']    # Transition log-probabilities  (T x T)
log_B   = hmm['log_B']    # Emission log-probabilities    (T x V)
log_pi  = hmm['log_pi']   # Initial log-probabilities     (T,)
tag2idx = hmm['tag2idx']
idx2tag = hmm['idx2tag']
word2idx= hmm['word2idx']
tagset  = hmm['tagset']

T = len(tagset)           # number of tags

print('✅ HMM loaded from hmm_model.pkl')
print(f'   Number of tags : {T}')
print(f'   Vocab size     : {len(word2idx):,}')

✅ HMM loaded from hmm_model.pkl
   Number of tags : 44
   Vocab size     : 10,109


---
## Cell 2 — Handle Unknown Words (OOV)

> 🧠 **The OOV problem.**  
> OOV = Out Of Vocabulary. Any word the model never saw during training.  
> If we just look it up and find nothing, the emission probability is undefined.
>
> **Solution — morphological heuristics:**  
> We look at the *shape* of the word to make an educated guess about its tag.  
> This is exactly what humans do with words they've never seen:  
> *"Frobulate"* — sounds like a verb. *"Frobulation"* — sounds like a noun.

In [ ]:
def get_word_idx(word):
    """
    Look up a word in the vocabulary.
    If unseen (OOV), return the index of <UNK>.
    """
    return word2idx.get(word.lower(), word2idx['<UNK>'])


def oov_log_probs(word):
    """
    For unknown words, return a log-probability boost vector of shape (T,).
    We add a small bonus to tags that morphologically match the word.
    This steers Viterbi toward the right tag even for unseen words.

    Rules learned from Penn Treebank patterns:
      -ing suffix  → likely VBG (gerund)
      -ed  suffix  → likely VBD (past tense)
      -ly  suffix  → likely RB  (adverb)
      -tion/-ness/-ment/-ity → likely NN (noun)
      -er/-est     → likely JJ  (comparative/superlative adjective)
      Starts with capital → likely NNP (proper noun)
      Ends in -s   → likely NNS (plural noun) or VBZ (3rd person verb)
    """
    boost = np.zeros(T)        # start with no boost for any tag
    w = word.lower()
    BONUS = 3.0                # log-space bonus (equivalent to ~20x more likely)

    def add_boost(tag_name):
        if tag_name in tag2idx:
            boost[tag2idx[tag_name]] += BONUS

    if w.endswith('ing'):                          add_boost('VBG')
    elif w.endswith('ed'):                         add_boost('VBD'); add_boost('VBN')
    elif w.endswith('ly'):                         add_boost('RB')
    elif w.endswith(('tion','ness','ment','ity')): add_boost('NN')
    elif w.endswith(('er','est')):                 add_boost('JJR'); add_boost('JJS')
    elif word[0].isupper():                        add_boost('NNP')
    elif w.endswith('s'):                          add_boost('NNS'); add_boost('VBZ')

    return boost


# Quick test
test_words = ['running', 'quickly', 'walked', 'London', 'cats']
print('🧪 OOV heuristic test:')
for w in test_words:
    boost = oov_log_probs(w)
    boosted_tags = [(idx2tag[i], boost[i]) for i in np.where(boost > 0)[0]]
    print(f'   "{w}" → boosted tags: {boosted_tags}')

🧪 OOV heuristic test:
   "running" → boosted tags: [('VBG', np.float64(3.0))]
   "quickly" → boosted tags: [('RB', np.float64(3.0))]
   "walked" → boosted tags: [('VBD', np.float64(3.0)), ('VBN', np.float64(3.0))]
   "London" → boosted tags: [('NNP', np.float64(3.0))]
   "cats" → boosted tags: [('NNS', np.float64(3.0)), ('VBZ', np.float64(3.0))]


---
## Cell 3 — The Viterbi Algorithm

> 🧠 **Two passes:**
> 1. **Forward pass** — fill the grid column by column, tracking the best score at each cell
> 2. **Backward pass** — start from the best final tag, follow the backpointer trail back to the start
>
> Everything is in log-space, so multiplications become additions.

In [ ]:
def viterbi(sentence):
    """
    Run the Viterbi algorithm on a sentence.

    Args:
        sentence : list of word strings  e.g. ['The', 'dog', 'runs']

    Returns:
        best_tags : list of POS tag strings  e.g. ['DT', 'NN', 'VBZ']
        best_score: float, the log-probability of the best path
    """
    N = len(sentence)    # number of words in the sentence

    # ── Grid initialisation ──
    # viterbi_grid[t][i] = best log-score of any path ending at tag i at position t
    # backpointer[t][i]  = which tag at position t-1 led to tag i at position t
    viterbi_grid = np.full((N, T), -np.inf)   # start everything at -infinity
    backpointer  = np.zeros((N, T), dtype=int)

    # ── Step 1: Initialise — first word ──
    word  = sentence[0]
    wi    = get_word_idx(word)
    emit  = log_B[:, wi]                        # emission scores for all tags
    if wi == word2idx['<UNK>']:                 # apply OOV boost if unknown
        emit = emit + oov_log_probs(word)

    viterbi_grid[0] = log_pi + emit             # π(tag) × P(word | tag)

    # ── Step 2: Forward pass — words 1 to N-1 ──
    for t in range(1, N):
        word = sentence[t]
        wi   = get_word_idx(word)
        emit = log_B[:, wi]
        if wi == word2idx['<UNK>']:
            emit = emit + oov_log_probs(word)

        # For every possible current tag j, find the best previous tag i
        # score = best_prev_score[i] + transition[i→j] + emission[j→word]
        for j in range(T):
            scores          = viterbi_grid[t-1] + log_A[:, j]   # shape (T,)
            best_prev       = np.argmax(scores)
            viterbi_grid[t][j] = scores[best_prev] + emit[j]
            backpointer[t][j]  = best_prev

    # ── Step 3: Find the best final tag ──
    best_last_tag = np.argmax(viterbi_grid[N-1])
    best_score    = viterbi_grid[N-1][best_last_tag]

    # ── Step 4: Backtrack to recover the full tag sequence ──
    best_path = [best_last_tag]
    for t in range(N-1, 0, -1):               # walk backwards
        best_path.append(backpointer[t][best_path[-1]])

    best_path.reverse()                        # flip to left→right order
    best_tags = [idx2tag[i] for i in best_path]

    return best_tags, best_score


print('✅ Viterbi function defined')

✅ Viterbi function defined


---
## Cell 4 — Tag Some Sentences!

Let's see the tagger in action on handcrafted examples.  
These are designed to test specific challenges — ambiguity, OOV words, modal verbs.

In [ ]:
def tag_and_display(sentence_str):
    """Helper: tokenise, tag, and pretty-print a sentence."""
    words = sentence_str.strip().split()
    tags, score = viterbi(words)

    print(f'  Sentence : {sentence_str}')
    print(f'  Tags     : {tags}')
    print(f'  Score    : {score:.2f}  (log-probability of best path)')

    # Aligned word/tag display
    pairs = '  '.join(f'{w}/{t}' for w, t in zip(words, tags))
    print(f'  Output   : {pairs}')
    print()


print('🏷️  Tagging test sentences:\n')

# Test 1: Basic sentence
print('── Test 1: Basic sentence ──')
tag_and_display('The dog runs quickly in the park')

# Test 2: Ambiguous word — "can" (modal verb vs noun)
print('── Test 2: Ambiguous word "can" ──')
tag_and_display('They can fish in the river')
tag_and_display('She opened the can carefully')

# Test 3: OOV words (not in training vocab)
print('── Test 3: OOV words ──')
tag_and_display('The frobulating contraption malfunctioned badly')

# Test 4: Proper nouns
print('── Test 4: Proper nouns ──')
tag_and_display('John quickly read the interesting book')

🏷️  Tagging test sentences:

── Test 1: Basic sentence ──
  Sentence : The dog runs quickly in the park
  Tags     : ['DT', 'NN', 'VBZ', 'RB', 'IN', 'DT', 'NN']
  Score    : -49.84  (log-probability of best path)
  Output   : The/DT  dog/NN  runs/VBZ  quickly/RB  in/IN  the/DT  park/NN

── Test 2: Ambiguous word "can" ──
  Sentence : They can fish in the river
  Tags     : ['PRP', 'MD', 'VB', 'IN', 'DT', 'NN']
  Score    : -40.15  (log-probability of best path)
  Output   : They/PRP  can/MD  fish/VB  in/IN  the/DT  river/NN

  Sentence : She opened the can carefully
  Tags     : ['PRP', 'VBD', 'DT', 'MD', 'RB']
  Score    : -39.97  (log-probability of best path)
  Output   : She/PRP  opened/VBD  the/DT  can/MD  carefully/RB

── Test 3: OOV words ──
  Sentence : The frobulating contraption malfunctioned badly
  Tags     : ['DT', 'JJ', 'NN', 'VBD', 'RB']
  Score    : -42.02  (log-probability of best path)
  Output   : The/DT  frobulating/JJ  contraption/NN  malfunctioned/VBD  badly/RB

─

---
## Cell 5 — Visualise the Viterbi Grid

> 🧠 This cell shows you the actual grid the algorithm fills.  
> Each row = a tag, each column = a word.  
> The highlighted path is what Viterbi chose as the best sequence.

In [ ]:
def visualise_grid(sentence_str, top_n_tags=8):
    """
    Show the Viterbi score grid for a short sentence.
    Only shows the top_n_tags most relevant tags for readability.
    """
    words = sentence_str.strip().split()
    N     = len(words)

    # Run Viterbi but also capture the grid
    viterbi_grid = np.full((N, T), -np.inf)
    backpointer  = np.zeros((N, T), dtype=int)

    wi   = get_word_idx(words[0])
    emit = log_B[:, wi]
    if wi == word2idx['<UNK>']: emit = emit + oov_log_probs(words[0])
    viterbi_grid[0] = log_pi + emit

    for t in range(1, N):
        wi   = get_word_idx(words[t])
        emit = log_B[:, wi]
        if wi == word2idx['<UNK>']: emit = emit + oov_log_probs(words[t])
        for j in range(T):
            scores = viterbi_grid[t-1] + log_A[:, j]
            bp     = np.argmax(scores)
            viterbi_grid[t][j] = scores[bp] + emit[j]
            backpointer[t][j]  = bp

    # Recover best path
    best_last = np.argmax(viterbi_grid[N-1])
    best_path = [best_last]
    for t in range(N-1, 0, -1):
        best_path.append(backpointer[t][best_path[-1]])
    best_path.reverse()

    # Select top_n_tags by their max score across the sentence
    tag_max_scores = viterbi_grid.max(axis=0)
    top_tags_idx   = np.argsort(tag_max_scores)[::-1][:top_n_tags]
    top_tags_idx   = sorted(top_tags_idx)   # sort for consistent display

    # Print grid
    col_w = 10
    header = f'  {"TAG":<6}' + ''.join(f'{w:>{col_w}}' for w in words)
    print(header)
    print('  ' + '-' * (6 + col_w * N))

    for ti in top_tags_idx:
        tag    = idx2tag[ti]
        in_path = [ti == best_path[t] for t in range(N)]
        row    = f'  {tag:<6}'
        for t in range(N):
            score = viterbi_grid[t][ti]
            cell  = f'{score:>8.1f}'
            if in_path[t]:
                cell = f'[{score:>6.1f}]'   # highlight best path
            row += f'{cell:>{col_w}}'
        print(row)

    print()
    best_tags = [idx2tag[i] for i in best_path]
    print(f'  Best path: {list(zip(words, best_tags))}')
    print('  (bracketed values = chosen path)')


print('📊 Viterbi Grid Visualisation\n')
visualise_grid('The dog runs fast')

📊 Viterbi Grid Visualisation

  TAG          The       dog      runs      fast
  ----------------------------------------------
  CC         -12.4     -19.7     -26.3     -37.3
  DT      [  -2.9]     -19.1     -28.2     -35.5
  IN         -11.8     -17.3     -24.9     -36.1
  JJ         -11.1     -14.2     -26.5     -35.5
  NNP        -10.0     -14.8     -25.5     -36.3
  PRP        -12.1     -20.4     -28.4     -37.4
  RB         -12.5     -17.1     -27.3  [ -34.2]
  ``         -11.9     -17.3     -28.7     -37.5

  Best path: [('The', 'DT'), ('dog', 'NN'), ('runs', 'VBZ'), ('fast', 'RB')]
  (bracketed values = chosen path)


---
## Cell 6 — Save the Tagger Function

We export the tagger as a standalone module so Step 04 (Evaluation) can import it cleanly.

In [ ]:
tagger_code = '''
import pickle
import numpy as np

with open('hmm_model.pkl', 'rb') as f:
    hmm = pickle.load(f)

log_A    = hmm['log_A']
log_B    = hmm['log_B']
log_pi   = hmm['log_pi']
tag2idx  = hmm['tag2idx']
idx2tag  = hmm['idx2tag']
word2idx = hmm['word2idx']
tagset   = hmm['tagset']
T        = len(tagset)

def get_word_idx(word):
    return word2idx.get(word.lower(), word2idx["<UNK>"])

def oov_log_probs(word):
    boost = np.zeros(T)
    w     = word.lower()
    BONUS = 3.0
    def add(tag):
        if tag in tag2idx: boost[tag2idx[tag]] += BONUS
    if w.endswith("ing"):                           add("VBG")
    elif w.endswith("ed"):                          add("VBD"); add("VBN")
    elif w.endswith("ly"):                          add("RB")
    elif w.endswith(("tion","ness","ment","ity")): add("NN")
    elif w.endswith(("er","est")):                  add("JJR"); add("JJS")
    elif word[0].isupper():                         add("NNP")
    elif w.endswith("s"):                           add("NNS"); add("VBZ")
    return boost

def viterbi(sentence):
    N            = len(sentence)
    viterbi_grid = np.full((N, T), -np.inf)
    backpointer  = np.zeros((N, T), dtype=int)
    wi   = get_word_idx(sentence[0])
    emit = log_B[:, wi]
    if wi == word2idx["<UNK>"]: emit = emit + oov_log_probs(sentence[0])
    viterbi_grid[0] = log_pi + emit
    for t in range(1, N):
        wi   = get_word_idx(sentence[t])
        emit = log_B[:, wi]
        if wi == word2idx["<UNK>"]: emit = emit + oov_log_probs(sentence[t])
        for j in range(T):
            scores             = viterbi_grid[t-1] + log_A[:, j]
            bp                 = np.argmax(scores)
            viterbi_grid[t][j] = scores[bp] + emit[j]
            backpointer[t][j]  = bp
    best_last = np.argmax(viterbi_grid[N-1])
    best_path = [best_last]
    for t in range(N-1, 0, -1):
        best_path.append(backpointer[t][best_path[-1]])
    best_path.reverse()
    return [idx2tag[i] for i in best_path], viterbi_grid[N-1][best_last]
'''

with open('tagger.py', 'w') as f:
    f.write(tagger_code)

print('✅ Saved → tagger.py')
print('   Step 04 will import this with: from tagger import viterbi')
print()
print('🚀 Ready for Step 04 — Evaluation & Accuracy')

✅ Saved → tagger.py
   Step 04 will import this with: from tagger import viterbi

🚀 Ready for Step 04 — Evaluation & Accuracy


---
## ✅ Step 03 Complete — Summary

| What we built | Detail |
|---|---|
| `get_word_idx()` | Safely maps any word to vocab index, unknown → `<UNK>` |
| `oov_log_probs()` | Morphological heuristics for unseen words |
| `viterbi()` | Full forward + backtrack algorithm in log-space |
| Grid visualiser | Shows exactly how Viterbi fills and chooses its path |
| `tagger.py` | Clean standalone module ready for evaluation |

**Next up → Step 04: Evaluation — accuracy, confusion matrix, error analysis**